In [6]:
import streamlit as st
import pandas as pd
import numpy as np
import requests
import time
import asyncio
import aiohttp
import logging
import plotly.express as px
from datetime import datetime
from functools import partial
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import cpu_count
from constants import api_key

In [ ]:
def get_current_temp_sync(city, api_key):
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()['main']['temp']
    else:
        print('Oh, something went wrong (as usually it is)')
        return None

async def get_current_temp_async(city, api_key):
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            if response.status == 200:
                data = await response.json()
                return data['main']['temp']
            else:
                print('Oh, something went wrong (as usually it is)')
            return None

In [10]:
get_current_temp_sync('Beijing', api_key)

-7.06

In [19]:
season_stats = pd.read_csv('normal_temperature_ranges.csv')
season_stats.head(20)

,city,season,min_temperature,max_temperature
0,Beijing,autumn,6.074720,26.042491
1,Beijing,spring,3.168096,22.662997
2,Beijing,summer,17.505122,36.991398
3,Beijing,winter,-12.176639,8.053121
4,Berlin,autumn,1.076114,21.304993
5,Berlin,spring,0.471116,20.334406
6,Berlin,summer,10.256252,29.710283
7,Berlin,winter,-9.601367,10.190230
8,Cairo,autumn,15.525829,34.735032
9,Cairo,spring,14.730824,35.423210


In [22]:
def check_temperature_status(city, current_temp, season_stats, season='winter'):
    city_stats = season_stats[(season_stats['city'] == city) & (season_stats['season'] == season)]
    row = city_stats.iloc[0]
    is_normal = row['min_temperature'] <= current_temp <= row['max_temperature']
    status = "common temperature" if is_normal else "anomaly"
    return {
        "city": city,
        "current_temp": current_temp,
        "season": season,
        "status": status
    }

In [23]:
cities = ["Berlin", "Cairo", "Dubai", "Beijing", "Moscow"]

# Тестируем за счет синхронного запуска
start = time.time()
for city in cities:
    temp = get_current_temp_sync(city, api_key)
    result = check_temperature_status(city, temp, season_stats)
    print(f"Город: {city}, температура: {result['current_temp']}, Аномалия или нет?: {result['status']}")
print(f"Потратили нашего драгоценного времени без асинхронщины: {time.time() - start:.2f}")

# Добавим ASSинхронщину!
start = time.time()
async def run_async_test():
    tasks = [get_current_temp_async(city, api_key) for city in cities]
    temps = await asyncio.gather(*tasks)
    for city, temp in zip(cities, temps):
        result = check_temperature_status(city, temp, season_stats)
        print(f"Город: {city}, температура: {result['current_temp']}, Аномалия или нет?: {result['status']}")

await run_async_test()
print(f"Потратили нашего драгоценного времени с асинхронщины: {time.time() - start:.2f}")

Город: Berlin, температура: 0.5, Аномалия или нет?: common temperature
Город: Cairo, температура: 17.42, Аномалия или нет?: common temperature
Город: Dubai, температура: 23.96, Аномалия или нет?: common temperature
Город: Beijing, температура: -8.06, Аномалия или нет?: common temperature
Город: Moscow, температура: -3.35, Аномалия или нет?: common temperature
Потратили нашего драгоценного времени без асинхронщины: 3.59
Город: Berlin, температура: 0.5, Аномалия или нет?: common temperature
Город: Cairo, температура: 17.42, Аномалия или нет?: common temperature
Город: Dubai, температура: 23.96, Аномалия или нет?: common temperature
Город: Beijing, температура: -8.06, Аномалия или нет?: common temperature
Город: Moscow, температура: -3.35, Аномалия или нет?: common temperature
Потратили нашего драгоценного времени с асинхронщины: 0.67


### Что лучше - решение в лоб или с помощью асинхронного программирования?

+ Решение в лоб проще с точки зрения написания и дебага (классика)

+ Асинхронщина быстрее, так как почти все время работы скрипта занимает пинг сервера, то есть если запросы идут параллельно на сервер - это сильно быстрее

+ Поэтому для масштабирования на большие системы асинхронный подход лучше, так как гораздо быстрее (что и подтверждается здесь, 0.67 секунд против 3.59 при обычном цикле по 5 городам)